In [16]:
%%writefile db_engine.py
import sys
import pandas as pd
import psycopg2

# Database Connection Parameters
# Replace 'YOUR_ACTUAL_PASSWORD' with your real PostgreSQL password
DB_PARAMS = {
    "dbname": "process_mining_db",
    "user": "postgres",
    "password": "Reshuadmin",
    "host": "localhost",
    "port": "5432"
}

def get_db_connection():
    """
    Establishes and returns a native psycopg2 DBAPI connection to PostgreSQL.
    """
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        return conn
    except Exception as e:
        print(f"[-] Database connection failure: {e}")
        sys.exit(1)

def fetch_transition_matrix(min_volume: int = 500) -> pd.DataFrame:
    """
    Executes vectorized window function query on p2p_event_logs
    and returns activity-to-activity transition metrics as a DataFrame.
    """
    conn = get_db_connection()
    
    query = f"""
    WITH SequencedEvents AS (
        SELECT 
            case_id,
            activity_name AS source_node,
            event_timestamp AS t1,
            LEAD(activity_name) OVER (
                PARTITION BY case_id 
                ORDER BY event_timestamp ASC
            ) AS target_node,
            LEAD(event_timestamp) OVER (
                PARTITION BY case_id 
                ORDER BY event_timestamp ASC
            ) AS t2
        FROM p2p_event_logs
        WHERE event_timestamp >= '2018-01-01 00:00:00'
          AND event_timestamp <= '2019-06-30 23:59:59'
    ),
    Transitions AS (
        SELECT 
            source_node,
            target_node,
            EXTRACT(EPOCH FROM (t2 - t1)) / 3600.0 AS delay_hours
        FROM SequencedEvents
        WHERE target_node IS NOT NULL
    )
    SELECT 
        source_node,
        target_node,
        COUNT(*) AS volume,
        ROUND(AVG(delay_hours)::numeric, 2) AS avg_delay_hours,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY delay_hours)::numeric, 2) AS median_delay_hours
    FROM Transitions
    GROUP BY source_node, target_node
    HAVING COUNT(*) >= {min_volume}
    ORDER BY volume DESC;
    """
    
    try:
        # Native psycopg2 connection + raw string query execution
        df = pd.read_sql_query(query, conn)
    finally:
        conn.close()
        
    return df

if __name__ == "__main__":
    df_result = fetch_transition_matrix(min_volume=500)
    print(f"[+] Successfully extracted {len(df_result)} aggregated transition pairs.")
    print(df_result.head(10))

Overwriting db_engine.py


In [17]:
import importlib
import db_engine

# Force module reload from disk
importlib.reload(db_engine)

# Execute test query
df_test = db_engine.fetch_transition_matrix(min_volume=1000)
display(df_test.head(10))

C:\Users\reshu\p2p_process_intelligence\db_engine.py:73: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


,source_node,target_node,volume,avg_delay_hours,median_delay_hours
0,Record Service Entry Sheet,Record Service Entry Sheet,139103,0.04,0.00
1,Record Invoice Receipt,Clear Invoice,134955,1049.33,865.45
2,Record Goods Receipt,Record Invoice Receipt,112183,519.27,307.13
3,Vendor creates invoice,Record Goods Receipt,110213,58.98,34.33
4,Create Purchase Order Item,Vendor creates invoice,103150,286.72,155.81
5,Vendor creates invoice,Record Invoice Receipt,95218,301.47,112.03
6,Record Goods Receipt,Vendor creates invoice,80616,79.14,14.83
7,Create Purchase Order Item,Record Goods Receipt,77206,369.23,306.33
8,Record Goods Receipt,Record Goods Receipt,54273,6.72,0.00
9,Remove Payment Block,Clear Invoice,49423,871.95,599.15
